In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_DIR = Path(
    "/Users/apple/Documents/team4-final-project/data/processed"
)


# 1. 전처리된 파일에서 계산에 필요한 컬럼 불러오기
# 사용자 ID의 비교 형식을 맞추기 위해 문자형으로 읽기

df_accounts_user = pd.read_csv(
    DATA_DIR / "accounts_user_preprocessed_2.csv",
    usecols=["id"],
    dtype="string",
)

df_accounts_friendrequest = pd.read_csv(
    DATA_DIR / "accounts_friendrequest_preprocessed_2.csv",
    usecols=["send_user_id", "receive_user_id"],
    dtype="string",
)

df_accounts_userquestionrecord = pd.read_csv(
    DATA_DIR / "accounts_userquestionrecord_preprocessed_2.csv",
    usecols=["chosen_user_id"],
    dtype="string",
)

df_accounts_pointhistory = pd.read_csv(
    DATA_DIR / "accounts_pointhistory_preprocessed_2.csv",
    usecols=["user_id"],
    dtype="string",
)

df_accounts_paymenthistory = pd.read_csv(
    DATA_DIR / "accounts_paymenthistory_preprocessed_2.csv",
    usecols=["user_id", "created_at"],
    dtype="string",
)


# 2. 사용자별 기간 내 첫 결제 시점
df_accounts_paymenthistory["created_at"] = pd.to_datetime(
    df_accounts_paymenthistory["created_at"],
    format="mixed",
)

first_payment = (
    df_accounts_paymenthistory
    .groupby("user_id", as_index=False)["created_at"]
    .min()
)


# 3. 행동별 고유 사용자 집합
# set은 같은 사용자 ID가 여러 번 있어도 한 번만 포함

signup_users = set(
    df_accounts_user["id"].dropna()
)

# 친구 수 조건 없이 발신자와 수신자를 모두 포함
friend_users = (
    set(df_accounts_friendrequest["send_user_id"].dropna())
    | set(df_accounts_friendrequest["receive_user_id"].dropna())
)

# 투표 수신자: chosen_user_id 사용
received_users = set(
    df_accounts_userquestionrecord["chosen_user_id"].dropna()
)

# 이미 delta_point < 0으로 전처리된 파일
point_decrease_users = set(
    df_accounts_pointhistory["user_id"].dropna()
)

payment_users = set(
    first_payment["user_id"]
)


# 4. 단계별 사용자 집합

# 전체 가입 사용자
step1 = signup_users

# 위 사용자 중 수락된 친구 요청의 발신자 또는 수신자
step2 = step1 & friend_users

# 위 사용자 중 투표를 받은 사용자
# 포인트 감소 여부는 조건에 포함하지 않음
step3 = step2 & received_users

# 위 사용자 중 기간 내 결제 기록이 있는 사용자
step4 = step3 & payment_users


# 5. 단계별 사용자 수와 직전 단계 대비 비율
funnel = pd.DataFrame({
    "단계": ["Step 1", "Step 2", "Step 3", "Step 4"],
    "기준": [
        "전체 가입 사용자",
        "수락 상태 친구 요청 기록 보유",
        "투표 수신 기록 보유",
        "기간 내 결제 기록 보유",
    ],
    "유저 수(명)": [
        len(step1),
        len(step2),
        len(step3),
        len(step4),
    ],
})

funnel["구간 비율(%)"] = (
    funnel["유저 수(명)"]
    / funnel["유저 수(명)"].shift(1)
    * 100
).round(2)

if len(step1) > 0:
    funnel.loc[0, "구간 비율(%)"] = 100.0

display(funnel)

print(
    "※ 동일 사용자의 기록 보유 여부를 기준으로 계산했습니다."
    "\n※ 행동의 시간 순서는 반영하지 않았습니다."
    "\n※ NaN은 분모가 0이어서 계산할 수 없는 경우입니다."
)


,단계,기준,유저 수(명),구간 비율(%)
0,Step 1,전체 가입 사용자,677075,100.00
1,Step 2,수락 상태 친구 요청 기록 보유,574163,84.80
2,Step 3,투표 수신 기록 보유,10647,1.85
3,Step 4,기간 내 결제 기록 보유,1041,9.78


※ 동일 사용자의 기록 보유 여부를 기준으로 계산했습니다.
※ 행동의 시간 순서는 반영하지 않았습니다.
※ NaN은 분모가 0이어서 계산할 수 없는 경우입니다.


In [3]:
import pandas as pd
from IPython.display import display


# 1. 전처리된 포인트·결제 기록 불러오기

df_accounts_pointhistory = pd.read_csv(
    DATA_DIR / "accounts_pointhistory_preprocessed_2.csv",
    usecols=["user_id", "created_at"],
    dtype={"user_id": "string"}
)

df_accounts_paymenthistory = pd.read_csv(
    DATA_DIR / "accounts_paymenthistory_preprocessed_2.csv",
    usecols=["user_id", "created_at"],
    dtype={"user_id": "string"}
)

for df in [
    df_accounts_pointhistory,
    df_accounts_paymenthistory
]:
    df["created_at"] = pd.to_datetime(
        df["created_at"],
        format="mixed"
    )


# 2. 재화 소모 사용자와 결제 사용자
# 포인트 파일은 이미 음수 기록만 남겨진 상태

spend_users = set(
    df_accounts_pointhistory["user_id"].dropna()
)

payment_users = set(
    df_accounts_paymenthistory["user_id"].dropna()
)


# 3. 기존 Step 1~3을 유지하고 새로운 Step 4~5 구성

# Step 4: 투표 수신자 중 재화 소모 기록 보유
loop1_step4 = step3 & spend_users

# Step 5: 위 사용자 중 결제 기록 보유
loop1_step5 = loop1_step4 & payment_users


# 4. 1차 흐름 결과

first_loop = pd.DataFrame({
    "단계": [
        "Step 1", "Step 2", "Step 3", "Step 4", "Step 5"
    ],
    "기준": [
        "전체 가입 사용자",
        "수락 상태 친구 요청 기록 보유",
        "투표 수신 기록 보유",
        "재화 소모 기록 보유",
        "기간 내 결제 기록 보유"
    ],
    "사용자 수": [
        len(step1),
        len(step2),
        len(step3),
        len(loop1_step4),
        len(loop1_step5)
    ]
})

previous_count = (
    first_loop["사용자 수"]
    .shift(1)
    .replace(0, float("nan"))
)

first_loop["직전 단계 대비(%)"] = (
    first_loop["사용자 수"] / previous_count * 100
).round(2)

if len(step1) > 0:
    first_loop.loc[0, "직전 단계 대비(%)"] = 100.0

display(first_loop)

,단계,기준,사용자 수,직전 단계 대비(%)
0,Step 1,전체 가입 사용자,677075,100.00
1,Step 2,수락 상태 친구 요청 기록 보유,574163,84.80
2,Step 3,투표 수신 기록 보유,10647,1.85
3,Step 4,재화 소모 기록 보유,3367,31.62
4,Step 5,기간 내 결제 기록 보유,358,10.63


In [4]:
# 1. 투표 수신 기록 불러오기

df_accounts_userquestionrecord = pd.read_csv(
    DATA_DIR / "accounts_userquestionrecord_preprocessed_2.csv",
    usecols=["chosen_user_id", "created_at"],
    dtype={"chosen_user_id": "string"}
)

df_accounts_userquestionrecord["created_at"] = pd.to_datetime(
    df_accounts_userquestionrecord["created_at"],
    format="mixed"
)


# 2. 사용자와 행동 시각 이름 정리

received_votes = df_accounts_userquestionrecord.rename(
    columns={
        "chosen_user_id": "user_id",
        "created_at": "received_at"
    }
)

point_spending = df_accounts_pointhistory.rename(
    columns={"created_at": "spent_at"}
)

payments = df_accounts_paymenthistory.rename(
    columns={"created_at": "payment_at"}
)


# 3. 1차 결제자들의 기준 결제 시각

base_payment = (
    payments
    .groupby("user_id", as_index=False)["payment_at"]
    .min()
    .rename(columns={"payment_at": "base_pay_at"})
)

base = pd.DataFrame(
    {"user_id": list(loop1_step5)},
    dtype="string"
).merge(
    base_payment,
    on="user_id",
    how="left"
)

if base["base_pay_at"].isna().any():
    raise ValueError(
        "일부 사용자의 기준 결제 시각을 찾지 못했습니다. "
        "결제 파일과 사용자 ID 형식을 확인하세요."
    )


# 4. 기준 결제 이후 투표 재수신

loop2_received = base.merge(
    received_votes,
    on="user_id",
    how="inner"
)

loop2_received = loop2_received[
    loop2_received["received_at"] > loop2_received["base_pay_at"]
]

# 기준 결제 이후의 첫 수신 기록
loop2_received = (
    loop2_received
    .sort_values("received_at")
    .drop_duplicates("user_id")
)


# 5. 투표 재수신 이후 재화 소모
# 특정 투표 ID와 연결하지 않고 사용자의 포인트 감소 기록 전체 사용

loop2_spent = loop2_received.merge(
    point_spending,
    on="user_id",
    how="inner"
)

loop2_spent = loop2_spent[
    loop2_spent["spent_at"] > loop2_spent["received_at"]
]

# 재수신 이후의 첫 소모 기록
loop2_spent = (
    loop2_spent
    .sort_values("spent_at")
    .drop_duplicates("user_id")
)


# 6. 재화 소모 이후 재결제

loop2_repaid = loop2_spent.merge(
    payments,
    on="user_id",
    how="inner"
)

loop2_repaid = loop2_repaid[
    loop2_repaid["payment_at"] > loop2_repaid["spent_at"]
]

# 소모 이후의 첫 추가 결제 기록
loop2_repaid = (
    loop2_repaid
    .sort_values("payment_at")
    .drop_duplicates("user_id")
    .rename(columns={"payment_at": "repay_at"})
)


# 7. 2차 흐름 결과

second_loop = pd.DataFrame({
    "단계": [
        "출발 집단",
        "2차 Step 1",
        "2차 Step 2",
        "2차 Step 3"
    ],
    "기준": [
        "1차 결제 기록 보유자",
        "기준 결제 이후 투표 재수신",
        "재수신 이후 재화 소모",
        "재화 소모 이후 재결제"
    ],
    "사용자 수": [
        len(base),
        len(loop2_received),
        len(loop2_spent),
        len(loop2_repaid)
    ]
})

previous_count = (
    second_loop["사용자 수"]
    .shift(1)
    .replace(0, float("nan"))
)

second_loop["직전 단계 대비(%)"] = (
    second_loop["사용자 수"] / previous_count * 100
).round(2)

if len(base) > 0:
    second_loop.loc[0, "직전 단계 대비(%)"] = 100.0

second_loop["1차 결제자 대비(%)"] = (
    second_loop["사용자 수"] / len(base) * 100
).round(2) if len(base) > 0 else float("nan")

display(second_loop)


# 8. 재결제자의 실제 행동 시각 확인

second_loop_detail = loop2_repaid[[
    "user_id",
    "base_pay_at",
    "received_at",
    "spent_at",
    "repay_at"
]]

display(second_loop_detail.head(10))

,단계,기준,사용자 수,직전 단계 대비(%),1차 결제자 대비(%)
0,출발 집단,1차 결제 기록 보유자,358,100.00,100.00
1,2차 Step 1,기준 결제 이후 투표 재수신,353,98.60,98.60
2,2차 Step 2,재수신 이후 재화 소모,334,94.62,93.30
3,2차 Step 3,재화 소모 이후 재결제,71,21.26,19.83


,user_id,base_pay_at,received_at,spent_at,repay_at
19,1205405,2023-05-14 00:33:10,2023-05-14 00:59:16,2023-05-14 04:54:23,2023-05-14 04:55:56
73,1151640,2023-05-14 10:06:37,2023-05-14 10:26:15,2023-05-14 10:44:00,2023-05-14 10:44:22
84,1187264,2023-05-14 09:21:06,2023-05-14 10:14:58,2023-05-14 12:54:08,2023-05-14 13:44:33
24,1158530,2023-05-14 05:01:39,2023-05-14 05:24:53,2023-05-14 05:25:11,2023-05-14 13:49:36
7,1211188,2023-05-13 23:12:52,2023-05-13 23:15:52,2023-05-14 02:33:40,2023-05-14 14:02:10
36,1217035,2023-05-14 05:45:13,2023-05-14 05:45:58,2023-05-14 05:46:40,2023-05-14 15:26:53
114,1183498,2023-05-14 11:56:48,2023-05-14 13:56:21,2023-05-14 16:58:24,2023-05-14 17:31:20
45,1237036,2023-05-14 03:06:58,2023-05-14 07:04:46,2023-05-14 07:18:35,2023-05-15 05:21:32
77,1137788,2023-05-14 08:48:56,2023-05-14 09:59:34,2023-05-14 11:33:39,2023-05-15 05:41:03
133,1274008,2023-05-15 03:43:42,2023-05-15 04:13:30,2023-05-15 04:22:56,2023-05-15 06:32:32
